# Train/test EC classification

Train a grouped graph classifier using the active manifest and completed graph cache. UniProt accession is the split group so a protein cannot occur in both train and test. Classes represented by a single UniProt accession are retained in training and reported as train-only. Protein graph vectorization shows a progress bar during training and prediction. XGBoost trains directly on sparse features with parallel histogram trees, with a progress bar for the 300 boosting rounds.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "notebooks" / "src" / "enzyme_workflow").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the repository root from this notebook's working directory")
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))
from enzyme_workflow.classifier import run_classifier


In [ ]:
# User parameters
TEST_SIZE = 0.20
RANDOM_STATE = 42
NODE_LABEL_ALPHABET = "aa20"  # aa20, hp2, charge4, physchem5, chem7, or dayhoff6
BASE_ESTIMATOR_KIND = "linear_sgd"  # xgboost, linear_sgd, or random_forest
MAX_SIMPLEX_DIMENSION = 3
MIN_PAIR_DISTANCE = 0
MAX_PAIR_DISTANCE = 4
NBITS = 11
VECTORIZER_PROGRESS = True
CLASSIFIER_PROGRESS = True
N_JOBS = -1  # use all available CPU cores for vectorization and classifier fitting


In [ ]:
classification = run_classifier(
    PROJECT_ROOT,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    node_label_alphabet=NODE_LABEL_ALPHABET,
    base_estimator_kind=BASE_ESTIMATOR_KIND,
    max_simplex_dimension=MAX_SIMPLEX_DIMENSION,
    min_pair_distance=MIN_PAIR_DISTANCE,
    max_pair_distance=MAX_PAIR_DISTANCE,
    nbits=NBITS,
    vectorizer_progress=VECTORIZER_PROGRESS,
    classifier_progress=CLASSIFIER_PROGRESS,
    n_jobs=N_JOBS,
)
print(f"Split: {classification.split_strategy}")
print(f"Accuracy: {classification.accuracy:.3f}; balanced accuracy: {classification.balanced_accuracy:.3f}")
print(f"Saved run: {classification.run_dir}")
display(classification.class_counts)
display(classification.class_group_counts)
if classification.train_only_classes:
    print("Train-only EC classes (single UniProt group):", classification.train_only_classes)
display(classification.train_counts)
display(classification.test_counts)
display(classification.report)
display(classification.confusion)
display(classification.confusion_figure)
display(classification.confusion_fractions)
display(classification.confusion_fraction_figure)
display(classification.roc_figure)
display(classification.pr_figure)
display(classification.predictions.head())


ROC and PR curves use micro-averaged one-vs-rest scores. The ROC dashed diagonal shows chance-level discrimination; the PR dashed horizontal line shows the positive-label prevalence baseline. Confusion fractions are normalized by true-class row, so each nonempty row sums to one. All four figures use square plotting axes.
